# Desafio técnico

### Fonte de dados: data/df_fraud_credit.csv

#### Realizando a leitura e importação de bibliotecas

In [27]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('data/df_fraud_credit.csv')

df.head()

,timestamp,sending_address,receiving_address,amount,transaction_type,location_region,ip_prefix,login_frequency,session_duration,purchase_pattern,age_group,risk_score,anomaly
0,1618185002,0x9d32d0bf2c00f41ce7ca01b66e174cc4dcb0c1da,0x39f82e1c09bc6d7baccc1e79e5621ff812f50572,67435.0,transfer,Europe,192.000,3,48,focused,established,18.75,low_risk
1,1698642474,0xd6e251c23cbf52dbd472f079147873e655d8096f,0x51e8fbe24f124e0e30a614e14401b9bbfed5384c,1.0,purchase,South America,172.000,5,61,focused,established,25.0,low_risk
2,1619180066,0x2e0925b922fed01f6a85d213ae2718f54b8ca305,0x52c7911879f783d590af45bda0c0ef2b8536706f,66211.0,purchase,Asia,192.168,3,74,focused,established,31.25,low_risk
3,1591413882,0x93efefc25fcaf31d7695f28018d7a11ece55457f,0x8ac3b7bd531b3a833032f07d4e47c7af6ea7bace,14998.0,transfer,South America,172.000,8,111,high_value,veteran,36.75,low_risk
4,1611257295,0xad3b8de45d63f5cce28aef9a82cf30c397c6ceb9,0x6fdc047c2391615b3facd79b4588c7e9106e49f2,66002.0,sale,Africa,172.160,6,100,high_value,veteran,62.5,moderate_risk


In [7]:
# Contagem de preenchimento

df.isnull().mean() * 100

timestamp            0.0
sending_address      0.0
receiving_address    0.0
amount               0.0
transaction_type     0.0
location_region      0.0
ip_prefix            0.0
login_frequency      0.0
session_duration     0.0
purchase_pattern     0.0
age_group            0.0
risk_score           0.0
anomaly              0.0
dtype: float64

In [18]:
df.count() # Cerca de 1Mi de registros

timestamp            1048575
sending_address      1048575
receiving_address    1048575
amount               1048575
transaction_type     1048575
location_region      1048575
ip_prefix            1048575
login_frequency      1048575
session_duration     1048575
purchase_pattern     1048575
age_group            1048575
risk_score           1048575
anomaly              1048575
dtype: int64

In [20]:
(df.astype(str).apply(lambda x: x.str.lower() == "none")).mean() * 100

timestamp            0.000000
sending_address      0.000000
receiving_address    0.000000
amount               0.537968
transaction_type     0.000000
location_region      0.000000
ip_prefix            0.000000
login_frequency      0.000000
session_duration     0.000000
purchase_pattern     0.000000
age_group            0.000000
risk_score           0.543118
anomaly              0.000000
dtype: float64

##### Existem alguns registros com "none" Ou seja, seriam nulos, mas vieram preenchidos com indicativo de nulo

### Como são poucos registros com "none" podemos dar drop das linhas

In [22]:
df = df[~df.astype(str).apply(lambda x: x.str.lower().eq("none")).any(axis=1)]

df.count()

timestamp            1037267
sending_address      1037267
receiving_address    1037267
amount               1037267
transaction_type     1037267
location_region      1037267
ip_prefix            1037267
login_frequency      1037267
session_duration     1037267
purchase_pattern     1037267
age_group            1037267
risk_score           1037267
anomaly              1037267
dtype: int64

## Caso o número de nulos fosse algo representativo, a ideia poderia ser um preenchimento com a média, e também acionar um alerta de qualidade de dados baixa

In [21]:
df.dtypes

timestamp              int64
sending_address       object
receiving_address     object
amount                object
transaction_type      object
location_region       object
ip_prefix            float64
login_frequency        int64
session_duration       int64
purchase_pattern      object
age_group             object
risk_score            object
anomaly               object
dtype: object

In [23]:
# Forçar schema corretamente (tratando a coluna 'amount')
df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0).astype(int)

# Agora o resto do schema pode ser aplicado normalmente
schema = {
    "timestamp": "int64",
    "sending_address": "string",
    "receiving_address": "string",
    # amount já foi tratado acima
    "transaction_type": "string",
    "location_region": "string",
    "ip_prefix": "string",
    "login_frequency": "int64",
    "session_duration": "int64",
    "purchase_pattern": "string",
    "age_group": "string",
    "risk_score": "float64",
    "anomaly": "string"
}

df = df.astype(schema)

print(df.dtypes)


C:\Users\Caio\AppData\Local\Temp\ipykernel_16376\4164736319.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0).astype(int)


timestamp                     int64
sending_address      string[python]
receiving_address    string[python]
amount                        int32
transaction_type     string[python]
location_region      string[python]
ip_prefix            string[python]
login_frequency               int64
session_duration              int64
purchase_pattern     string[python]
age_group            string[python]
risk_score                  float64
anomaly              string[python]
dtype: object


### Com os tipos de dados já devidamente ajustados, vamos avaliar os padrões das colunas e avaliar se há dados inconsistentes

In [31]:
def analyze_string_length(df, cols=None, exclude=None, na_as_len=np.nan, verbose=False):
    """
    Analisa comprimento de caracteres nas colunas string do df.
    Args:
        df: pandas DataFrame.
        cols: lista opcional de colunas a analisar (se None, detecta colunas 'object' ou 'string').
        exclude: lista opcional de colunas a ignorar (ex: ["location_region"]) ou string "location_region".
        na_as_len: se != np.nan, substitui NAs por esse comprimento (ex: 0). Se np.nan, ignora NAs nos cálculos.
        verbose: se True, retorna também DataFrame de flags por linha (col_below, col_above).
    Returns:
        summary_df: DataFrame com estatísticas por coluna (mean_len, std_len, low_thr, high_thr, %_below, %_above, ...).
        (opcional) flags: DataFrame booleano com colunas "<col>_below" e "<col>_above" indicando linhas fora do intervalo.
    """
    # Normalizar exclude para lista (aceita string ou lista)
    if exclude is None:
        exclude_list = []
    elif isinstance(exclude, str):
        exclude_list = [exclude]
    else:
        exclude_list = list(exclude)

    # detectar colunas string por padrão (object ou pandas StringDtype)
    if cols is None:
        cols = [
            c for c in df.columns
            if df[c].dtype == "object" or str(df[c].dtype).startswith("string")
        ]
    else:
        # manter apenas colunas existentes e avisar sobre as que faltam
        missing = [c for c in cols if c not in df.columns]
        if missing:
            print(f"Warning: as seguintes colunas não existem no DataFrame e serão ignoradas: {missing}")
        cols = [c for c in cols if c in df.columns]

    # aplicar exclusões
    if exclude_list:
        cols = [c for c in cols if c not in exclude_list]

    if not cols:
        raise ValueError("Nenhuma coluna válida para analisar (cols vazia após filtragem/exclusão).")

    summary = []
    flags = pd.DataFrame(index=df.index)

    for c in cols:
        # usar StringDtype para preservar pd.NA
        s = df[c].astype("string")
        length = s.str.len()  # resulta em Int64 (nullable) ou float se NA present

        # tratar NAs se solicitado
        if not np.isnan(na_as_len):
            length = length.fillna(na_as_len)

        mean = length.mean(skipna=True)
        std = length.std(skipna=True)

        # se std for nan (ex: todas NAs ou um único valor), transforme em 0 para evitar nan thresholds
        if np.isnan(std):
            std = 0.0

        if np.isnan(mean):
            mean = 0.0

        low_thr = mean - 2 * std
        high_thr = mean + 2 * std

        below_mask = length < low_thr
        above_mask = length > high_thr

        pct_below = below_mask.sum() / len(df) * 100
        pct_above = above_mask.sum() / len(df) * 100

        summary.append({
            "column": c,
            "mean_len": float(mean),
            "std_len": float(std),
            "low_thr": float(low_thr),
            "high_thr": float(high_thr),
            "%_below": float(pct_below),
            "%_above": float(pct_above),
            "count_below": int(below_mask.sum()),
            "count_above": int(above_mask.sum()),
            "n_non_null": int(length.notna().sum()),
            "n_rows": len(df)
        })

        # preencher flags (NaNs => False)
        flags[c + "_below"] = below_mask.fillna(False)
        flags[c + "_above"] = above_mask.fillna(False)

    summary_df = pd.DataFrame(summary).set_index("column")

    if verbose:
        return summary_df, flags
    return summary_df

cols_classes = ["location_region", "transaction_type", "purchase_pattern", "age_group", "anomaly"]

summary = analyze_string_length(df, cols=None, exclude=cols_classes, na_as_len=np.nan, verbose=False)
print(summary)

# Se quiser também obter as linhas que estão fora do padrão em qualquer coluna (verbose=True)
summary, flags = analyze_string_length(df, cols=None, exclude=cols_classes, na_as_len=np.nan, verbose=True)

# linhas com qualquer flag True:
rows_with_any_outlier = flags.any(axis=1)
print("\nPercentual de linhas com ao menos um campo string fora de 2 desvpad:", rows_with_any_outlier.mean() * 100)

                    mean_len   std_len    low_thr   high_thr  %_below  \
column                                                                  
sending_address    42.000000  0.000000  42.000000  42.000000      0.0   
receiving_address  42.000000  0.000000  42.000000  42.000000      0.0   
ip_prefix           5.401838  1.019541   3.362756   7.440919      0.0   

                   %_above  count_below  count_above  n_non_null   n_rows  
column                                                                     
sending_address        0.0            0            0     1037267  1037267  
receiving_address      0.0            0            0     1037267  1037267  
ip_prefix              0.0            0            0     1037267  1037267  

Percentual de linhas com ao menos um campo string fora de 2 desvpad: 0.0


In [32]:
cols_classes = ["location_region", "transaction_type", "purchase_pattern", "age_group", "anomaly"]

for col in cols_classes:
    if col in df.columns:
        print(f"\n=== {col} ===")
        print(df[col].value_counts(dropna=False))


=== location_region ===
location_region
North America    207879
Europe           207514
Asia             206431
South America    205562
Africa           204158
0                  5723
Name: count, dtype: Int64

=== transaction_type ===
transaction_type
sale        330410
purchase    329165
transfer    291962
scam         52145
phishing     33585
Name: count, dtype: Int64

=== purchase_pattern ===
purchase_pattern
high_value    348696
random        345023
focused       343548
Name: count, dtype: Int64

=== age_group ===
age_group
veteran        348696
new            345023
established    343548
Name: count, dtype: Int64

=== anomaly ===
anomaly
low_risk         837930
moderate_risk    113607
high_risk         85730
Name: count, dtype: Int64


### Mais um ponto para ser avaliado, contar o percentual de mal preenchimento para colunas de classes esperadas

#### Exemplo:

location_region

North America    207879

Europe           207514

Asia             206431

South America    205562

Africa           204158

0                  5723

String "0" não representa região

In [33]:
mask_zero = df["location_region"].astype(str).str.strip() == "0"
num_zero = mask_zero.sum()
total_rows = len(df)
pct_zero = (num_zero / total_rows) * 100

print(f"Linhas com location_region == '0': {num_zero} ({pct_zero:.2f}% do total)")

Linhas com location_region == '0': 5723 (0.55% do total)


#### Como novamente são poucos registros, vamos apenas eliminar as linhas com problemas

In [34]:
df = df[~mask_zero].copy()

print(f"\nNovo total de linhas: {len(df)} (redução de {pct_zero:.2f}%)")


Novo total de linhas: 1031544 (redução de 0.55%)


## Criando a tabela resultado

In [47]:
df_proc = df.copy()

df_proc['risk_score'] = pd.to_numeric(df_proc.get('risk_score'), errors='coerce')
df_proc['amount'] = pd.to_numeric(df_proc.get('amount'), errors='coerce')
df_proc['timestamp'] = pd.to_numeric(df_proc.get('timestamp'), errors='coerce').astype('Int64')

# 1) Tabela: location_region por média de risk_score (decrescente)
result1 = (
    df_proc
    .dropna(subset=['location_region'])   # opcional: remove linhas sem região (se preferir, remova esta linha)
    .groupby('location_region', dropna=False)
    .agg(
        mean_risk_score = ('risk_score', 'mean')
    )
    .reset_index()
)

result1 = result1.sort_values('mean_risk_score', ascending=False).reset_index(drop=True)

result1

,location_region,mean_risk_score
0,North America,45.163474
1,South America,45.135298
2,Asia,44.989270
3,Africa,44.909799
4,Europe,44.601177


In [50]:
mask_sale = df_proc['transaction_type'].astype('string').str.lower() == 'sale'
sales = df_proc[mask_sale].copy()

# Row number
sales_sorted = sales.sort_values(['receiving_address', 'timestamp'], ascending=[True, False])
sales_sorted = sales_sorted.assign(
    row_number = sales_sorted.groupby('receiving_address').cumcount() + 1
)

# pegar apenas a transação mais recente por receiving_address (row_number == 1)
most_recent_per_receiver = sales_sorted[sales_sorted['row_number'] == 1].copy()

# agora ordenar por amount decrescente e pegar top 3
result2 = (
    most_recent_per_receiver
    .sort_values('amount', ascending=False)
    .loc[:, ['receiving_address', 'amount', 'timestamp']]
    .reset_index(drop=True)
)

# exibir resultado
print("Top 3 receiving_address (entre transações mais recentes por receiving_address com transaction_type=='sale'):")
result2.head(3)

Top 3 receiving_address (entre transações mais recentes por receiving_address com transaction_type=='sale'):


,receiving_address,amount,timestamp
0,0xfe2650f030f2c966775e11009cb015e8852ecf4b,76757,1704177853
1,0xe37126a5b0724737b516b97d3fb92311021e235d,76716,1703848668
2,0x646142948a3add1d05edb7789d73a14ef261e109,76667,1704125654


### Exportando informações finais

In [51]:
# CSV
result1.to_csv("result1_risk_score_por_regiao.csv", index=False)
result2.to_csv("result2_top3_receiving_sale.csv", index=False)

## Claro que estamos tratando de um exemplo simples, com 1 Mi de linhas e poucas colunas
### tratando-se da realidade, com números de até bilhões de registros, nem sonharíamos em rodar isso em PANDAS
### Utilizaríamos Spark em uma sessão AWS Glue (ou um EMR) com acesso a catálogo de dados via lake formation
### Os dados finais salvos poderiam ser tabelas dentro do catálogo de dados, produtivas ou apenas em conta consumer! Podendo atualizar painéis informativos
### No caso de dados agrupados, gerando poucas linhas, como RESULT1, o processo poderia atualizar uma planilha em um S3 ou então diretamente em uma pasta OneDrive.